# Orthogonal Gradient Boosting

Import necessary libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
import numpy as np
import scipy.optimize

from testdata import diblock_mvn_sample
x_4_100 = diblock_mvn_sample(100, seed=0)
y_4_100 = np.random.default_rng(seed=0).normal(size=100)

x_4_200 = diblock_mvn_sample(200, seed=0)
y_4_200 = np.random.default_rng(seed=0).normal(size=200)

x_4_400 = diblock_mvn_sample(400, seed=0)
y_4_400 = np.random.default_rng(seed=0).normal(size=400)

x_4_800 = diblock_mvn_sample(800, seed=0)
y_4_800 = np.random.default_rng(seed=0).normal(size=800)


The following cell defines the class for Orthogonal Gradient Boosting objective function.

In [2]:
import numpy as np
from numba import njit
from numba.experimental import jitclass
from numba.types import int64, float64
from optikon import Propositionalization

@njit
def argsort_columns(x):
    n, p = x.shape
    out = np.empty((n, p), dtype=np.int64)
    for j in range(p):
        out[:, j] = np.argsort(x[:, j])
    return out


@jitclass
class OrthogonalGradientBoosting:
    g: float64[:]           # Original gradient vector
    g_bot: float64[:]       # Orthogonal projection of gradient (g_⊥)
    O: float64[:,:]         # Orthonormal basis of selected queries (n x t)
    epsilon: float64        # Regularization parameter (ε) 
    t: int64               # Number of already selected queries
    n: int64               # Number of data points
    
    # Incremental computation variables
    cum_sum_g: float64              # Cumulative sum of g_⊥
    cum_proj: float64[:]           # Cumulative projections onto basis vectors (length t)
    count: int64            # Current count for incremental computation
    
    # Interface for greedy_maximization
    value: float64          # Current objective value
    value_removed: float64  # Objective for left split (removed points)
    value_remaining: float64 # Objective for right split (remaining points)
    
    # Working variables for split computation
    total_cum_sum_g: float64        # Total cum_sum_g for current support
    total_cum_proj: float64[:]     # Total cum proj for current support
    total_count: int64      # Total count for current support

    def __init__(self, g, O=None, epsilon=1e-6):
        """
        Initialize orthogonal gradient boosting objective.
        
        Args:
            g: Gradient vector (n,)
            O: Orthonormal basis of already selected queries (n, t). If None, assumes no queries selected yet.
            epsilon: Regularization parameter to avoid division by zero
        """
        self.g = g.copy()
        self.n = len(g)
        self.epsilon = epsilon
        
        if O is None or O.shape[1] == 0:
            # No queries selected yet, orthogonal projection is identity
            self.t = 0
            self.O = np.zeros((self.n, 0), dtype=np.float64)
            self.g_bot = g.copy()
        else:
            self.t = O.shape[1]
            self.O = O.copy()
            # Compute orthogonal projection: g_⊥ = g - O O^T g
            self.g_bot = self._project_orthogonal(g)
        
        # Initialize working arrays
        max_t = max(self.t, 1)  # Ensure at least size 1 for numba
        self.cum_proj = np.zeros(max_t)
        self.total_cum_proj = np.zeros(max_t)
        self.cum_sum_g = 0.0
        self.count = 0
        
        # Initialize interface variables
        self.value = 0.0
        self.value_removed = 0.0
        self.value_remaining = 0.0
        self.total_cum_sum_g = 0.0
        self.total_count = 0

    def _project_orthogonal(self, v):
        """Project vector v onto orthogonal complement of range(O)."""
        if self.t == 0:
            return v.copy()
        v_bot = v.copy()
        for k in range(self.t):
            dot_product = 0.0
            for i in range(self.n):
                dot_product += self.O[i, k] * v[i]
            
            for i in range(self.n):
                v_bot[i] -= dot_product * self.O[i, k]
        
        return v_bot

    def _compute_objective(self, cum_sum_g, cum_proj, count):
        """Compute objective value from state variables."""
        if count <= 0:
            return 0.0
            
        q_norm_sq = float(count)
        
        q_bot_norm_sq = q_norm_sq
        for k in range(self.t):
            q_bot_norm_sq -= cum_proj[k] * cum_proj[k]
        
        if q_bot_norm_sq > 0:
            return abs(cum_sum_g) / (np.sqrt(q_bot_norm_sq) + self.epsilon)
        else:
            return 0.0
    
    def reset(self):
        """Reset for incremental computation over current support."""       
        # Reset incremental computation variables
        self.cum_sum_g = 0.0
        self.count = 0
        for k in range(self.t):
            self.cum_proj[k] = 0.0

    def incremental_add_point(self, point_idx):
        """
        Add a point to the current query being built incrementally.
        This implements the efficient computation from Algorithm 3.
        
        Returns the current objective value for the query built so far.
        """
        self.cum_sum_g += self.g_bot[point_idx]
        for k in range(self.t):
            self.cum_proj[k] += self.O[point_idx, k]
        
        # Increment count
        self.count += 1
        # Return objective value
        return self._compute_objective(self.cum_sum_g, self.cum_proj, self.count)

    
    def support(self, indices):
        """Set the current support and compute total statistics."""
        self.total_count = len(indices)
        self.total_cum_sum_g = 0.0
        
        # Reset total_cum_proj
        for k in range(self.t):
            self.total_cum_proj[k] = 0.0
        
        # Compute totals for current support
        for i in range(len(indices)):
            point_idx = indices[i]
            self.total_cum_sum_g += self.g_bot[point_idx]
            for k in range(self.t):
                self.total_cum_proj[k] += self.O[point_idx, k]
        
        # Initialize: no points removed yet, all remaining
        self.reset()  # Left side (removed) starts empty
        
        # Compute initial objective values
        self.value_remaining = self._compute_objective(self.total_cum_sum_g, self.total_cum_proj, self.total_count)
        self.value = self.value_remaining
        self.value_removed = 0.0

    def remove(self, point_idx):
        """Remove a point from remaining to removed, updating objectives."""
        # Add point to left side (removed)
        self.value_removed = self.incremental_add_point(point_idx)
        
        # Compute right side (remaining) = total - left
        cum_sum_g_right = self.total_cum_sum_g - self.cum_sum_g
        count_right = self.total_count - self.count
        
        # Compute cum_proj_right = total_cum_proj - cum_proj
        if count_right > 0:
            cum_proj_right = np.zeros(max(self.t, 1))
            for k in range(self.t):
                cum_proj_right[k] = self.total_cum_proj[k] - self.cum_proj[k]
            self.value_remaining = self._compute_objective(cum_sum_g_right, cum_proj_right, count_right)
        else:
            self.value_remaining = 0.0
    
    def compute_objective(self, q):
        """Optimized orthogonal gradient boosting objective."""
        # Direct dot product (no need to project q)
        dot_product = np.dot(self.g_bot, q)  
        
        # Efficient norm computation using Proposition 3
        q_norm_sq = np.dot(q, q)
        
        q_perp_norm_sq = q_norm_sq
        for k in range(self.t):
            projection_k = np.dot(self.O[:, k], q)  # O[:,k]^T q
            q_perp_norm_sq -= projection_k * projection_k
        
        q_perp_norm = np.sqrt(max(0, q_perp_norm_sq)) 
        
        return abs(dot_product) / (q_perp_norm + self.epsilon)

    def compute_split_objectives_incremental(self, support_indices, feature_order):
        """
        Compute objectives for all possible splits efficiently using incremental computation.
        
        Args:
            support_indices: Indices of current support
            feature_order: Ordering of support indices by feature value
            
        Returns:
            Arrays of (left_objectives, right_objectives) for each possible split
        """
        support_size = len(support_indices)
        if support_size <= 1:
            return np.zeros(0), np.zeros(0)
        
        left_objectives = np.zeros(support_size - 1)
        right_objectives = np.zeros(support_size - 1)
        
        # Compute all left objectives incrementally
        self.reset()
        for i in range(support_size - 1):
            point_idx = support_indices[feature_order[i]]
            left_objectives[i] = self.incremental_add_point(point_idx)
        
        # Compute all right objectives incrementally (in reverse order)
        self.reset()
        for i in range(support_size - 1, 0, -1):
            point_idx = support_indices[feature_order[i]]
            right_objectives[i-1] = self.incremental_add_point(point_idx)
        
        return left_objectives, right_objectives

    def add_query(self, q):
        """
        Add a new query to the orthonormal basis and update projections.
        This should be called after selecting the best query in a boosting iteration.
        
        Returns:
            True if query was added (not redundant), False otherwise
        """
        q_bot = self._project_orthogonal(q)
        q_bot_norm_sq = 0.0
        for i in range(self.n):
            q_bot_norm_sq += q_bot[i] * q_bot[i]
        q_bot_norm = np.sqrt(q_bot_norm_sq)
        
        if q_bot_norm > self.epsilon:  # Only add if not redundant
            # Create new orthonormal basis vector
            o_new = np.zeros(self.n)
            for i in range(self.n):
                o_new[i] = q_bot[i] / q_bot_norm
            
            # Extend orthonormal basis
            O_new = np.zeros((self.n, self.t + 1))
            for i in range(self.n):
                for j in range(self.t):
                    O_new[i, j] = self.O[i, j]
                O_new[i, self.t] = o_new[i]
            
            self.O = O_new
            self.t += 1
            
            # Resize cum_proj arrays
            cum_proj_new = np.zeros(self.t)
            for k in range(self.t - 1):
                cum_proj_new[k] = self.cum_proj[k]
            self.cum_proj = cum_proj_new
            
            total_cum_proj_new = np.zeros(self.t)
            for k in range(self.t - 1):
                total_cum_proj_new[k] = self.total_cum_proj[k]
            self.total_cum_proj = total_cum_proj_new
            
            # Update orthogonal projection of gradient
            self.g_bot = self._project_orthogonal(self.g)
            
            return True
        else:
            return False  # Query was redundant

    def get_basis(self):
        """Return the current orthonormal basis matrix."""
        return self.O.copy()

    def get_orthogonal_gradient(self):
        """Return the current orthogonal projection of the gradient."""
        return self.g_bot.copy()
    





The next cell is the greedy search function.

In [3]:
@njit
def greedy_maximization(x, obj, max_depth=5):
    n, p = x.shape
    orders = argsort_columns(x)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    cum_support_count = 0
    non_separable = 0

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=np.int64) # cursor buffer for order updates
    
    best_value = obj.value
    num_cond = 0

    for k in range(1, max_depth+1):
        cum_support_count += support_count

        obj.support(orders[:support_count, 0])
        
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):

            obj.reset()
            
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                obj.remove(orders[i, j])
                if x[orders[i, j], j]==x[orders[i+1, j], j]:
                    non_separable += 1
                    continue

                if obj.value_removed > best_value:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_value = obj.value_removed
                    improvement = True
                elif obj.value_remaining > best_value:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_value = obj.value_remaining
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = best_s*(x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised?
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1

    res = Propositionalization(v[:num_cond], t[:num_cond], s[:num_cond])
    return res, best_value, {'cum_support_count': cum_support_count,
                           'non_separable': non_separable}


The next cell defines the functions for calculating the weight vector for all the queries.

In [4]:
def _get_risk(labels, q_matrix, loss_type, lambda_reg):
    """Risk function for different loss types."""
    def risk_func(weights):
        predictions = q_matrix @ weights
        if loss_type == 'squared':
            return np.mean((predictions - labels) ** 2) + lambda_reg * np.sum(weights[1:] ** 2)
        elif loss_type == 'logistic':
            return np.mean(np.log(1 + np.exp(-labels * predictions))) + lambda_reg * np.sum(weights[1:] ** 2)
        elif loss_type == 'poisson':
            return np.mean(np.exp(predictions) - labels * predictions) + lambda_reg * np.sum(weights[1:] ** 2)
    return risk_func

def _get_gradient(labels, q_matrix, loss_type, lambda_reg):
    """Gradient function for different loss types."""
    def gradient_func(weights):
        predictions = q_matrix @ weights
        n_samples = len(labels)
        
        if loss_type == 'squared':
            residuals = predictions - labels
            grad = 2 * q_matrix.T @ residuals / n_samples
        elif loss_type == 'logistic':
            exp_term = np.exp(-labels * predictions)
            sigmoid_term = labels * exp_term / (1 + exp_term)
            grad = -q_matrix.T @ sigmoid_term / n_samples
        elif loss_type == 'poisson':
            grad = q_matrix.T @ (np.exp(predictions) - labels) / n_samples
        
        # Add regularization (don't regularize intercept)
        reg_grad = np.zeros_like(weights)
        reg_grad[1:] = 2 * lambda_reg * weights[1:]
        return grad + reg_grad
    return gradient_func

def _get_hessian(labels, q_matrix, loss_type, lambda_reg):
    """Hessian function for different loss types."""
    def hessian_func(weights):
        predictions = q_matrix @ weights
        n_samples = len(labels)
        
        if loss_type == 'squared':
            # Hessian: 2 * Q^T Q / n
            H = 2 * q_matrix.T @ q_matrix / n_samples
        elif loss_type == 'logistic':
            exp_term = np.exp(-labels * predictions)
            diag_weights = exp_term / ((1 + exp_term) ** 2)
            H = q_matrix.T @ np.diag(diag_weights) @ q_matrix / n_samples
        elif loss_type == 'poisson':
            diag_weights = np.exp(predictions)
            H = q_matrix.T @ np.diag(diag_weights) @ q_matrix / n_samples
        
        # Add regularization
        reg_H = np.zeros_like(H)
        np.fill_diagonal(reg_H[1:, 1:], 2 * lambda_reg)
        return H + reg_H
    return hessian_func

The code for generating rule ensembles using OGB

In [5]:
from sklearn.metrics import mean_squared_error

def generate_rule_ensemble(X, y, max_rules=10, max_depth=3, epsilon=1e-6, 
                          lambda_reg=0.01, loss_type='squared', verbose=True):
    """
    Generate a complete rule ensemble using Orthogonal Gradient Boosting.
    
    This implements the core corrective orthogonal boosting algorithm using
    the existing OrthogonalGradientBoosting class.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        y: Target vector (n_samples,)
        max_rules: Maximum number of rules to generate
        max_depth: Maximum depth per rule
        epsilon: Regularization for orthogonal objective
        lambda_reg: L2 regularization for weight fitting
        verbose: Print progress information
    
    Returns:
        dict with:
            'rules': List of selected rules (Propositionalization objects)
            'weights': Rule weights (including intercept at index 0)
            'predictions': Final predictions on training data
            'history': Training history information
    """
    
    if verbose:
        print(f"Generating rule ensemble: max_rules={max_rules}, max_depth={max_depth}")
        print("=" * 60)
    
    n_samples = len(y)
    
    # Initialize ensemble components
    rules = []
    intercept = np.mean(y)  # Start with mean as intercept
    
    # Initialize orthogonal gradient boosting
    ogb = OrthogonalGradientBoosting(y - intercept, epsilon=epsilon)
    
    # Training history
    history = {
        'iteration': [],
        'objective_values': [],
        'train_losses': [],
        'rule_descriptions': [],
        'weights': []
    }
    
    if verbose:
        initial_loss = np.mean((y - intercept) ** 2)
        print(f"Initial loss (intercept only): {initial_loss:.6f}")
        print(f"Initial intercept: {intercept:.4f}")
        print()
    
    # Main boosting loop
    for iteration in range(1, max_rules + 1):
        if verbose:
            print(f"Iteration {iteration}:")
        # if iteration >1:
        #     print('basis', residuals)
        # Compute current predictions and residuals
        current_pred = np.full(n_samples, intercept)
        for rule, weight in zip(rules, history['weights']):
            if len(history['weights']) > 0:  # Check if we have weights from previous iterations
                # rule_output = rule.support_all(X).astype(float)
                rule_support_indices = rule.support_all(X)  
                rule_output = np.zeros(n_samples)           
                rule_output[rule_support_indices] = 1.0 
                current_pred += weight * rule_output
        
        residuals = y - current_pred
        
        if verbose:
            current_loss = np.mean(residuals ** 2)
            print(f"  Current loss: {current_loss:.6f}")
            print(f"  Residual norm: {np.linalg.norm(residuals):.6f}")
        
        # Find best rule using orthogonal gradient boosting
        ogb= OrthogonalGradientBoosting(residuals, ogb.get_basis() if ogb.t > 0 else None, epsilon)
        rule, objective_value, stats = greedy_maximization(X, ogb, max_depth)
        
        if objective_value <= epsilon:
            if verbose:
                print(f"  Stopping: objective too small ({objective_value:.2e})")
            break
        
        # Add rule to ensemble
        rules.append(rule)
        
        # Corrective weight update - refit all weights
        rule_outputs = []
        for r in rules:
            rule_support_indices = r.support_all(X)  
            rule_output = np.zeros(n_samples)           
            rule_output[rule_support_indices] = 1.0 
            rule_outputs.append(rule_output)
        
        q_matrix = np.column_stack([np.ones(n_samples)] + rule_outputs)

        # Set up optimization functions
        risk_func = _get_risk(y, q_matrix, loss_type, lambda_reg)
        gradient_func = _get_gradient(y, q_matrix, loss_type, lambda_reg)
        hessian_func = _get_hessian(y, q_matrix, loss_type, lambda_reg)

        # Initialize weights: previous weights + 0 for new rule
        if len(rules) == 1:
            w_init = np.array([intercept, 0.0])
        else:
            w_init = np.append([intercept] + current_weights, 0.0)

        # Optimize using Newton-CG
        result = scipy.optimize.minimize(
            risk_func, w_init, 
            method='Newton-CG', 
            jac=gradient_func, 
            hess=hessian_func,
            options={'disp': False, 'xtol': 1e-6, 'maxiter': 100}
        )

        # Update intercept and weights
        all_coeffs = result.x
        intercept = all_coeffs[0]
        current_weights = all_coeffs[1:].tolist()
        
        # Update orthonormal basis
        rule_support_indices = rule.support_all(X)
        query_vector = np.zeros(n_samples)
        query_vector[rule_support_indices] = 1.0
        added = ogb.add_query(query_vector)
        
        if not added and verbose:
            print(f"  Warning: Rule was redundant!")
        
        # Compute predictions and loss
        final_pred = q_matrix @ all_coeffs
        final_loss = np.mean((y - final_pred) ** 2)
        
        # Record history
        rule_complexity = len(rule.support_all(X))
        history['iteration'].append(iteration)
        history['objective_values'].append(objective_value)
        history['train_losses'].append(final_loss)
        history['rule_descriptions'].append(rule.as_conj_str())
        history['weights'] = current_weights.copy()
        if verbose:
            print(f"  Current Rule Ensemble (Iteration {iteration}):")
            print(f"    Intercept: {intercept:.4f}")
            for i, (rule, weight) in enumerate(zip(rules, current_weights)):
                sign = "+" if weight >= 0 else ""
                print(f"    Rule {i+1}: {sign}{weight:.4f} if {rule.as_conj_str()}")
            print()
        
        if verbose:
            print(f"  Rule: {rule.as_conj_str()}")
            print(f"  Objective: {objective_value:.6f}")
            print(f"  Weight: {current_weights[-1]:.4f}")
            print(f"  Support size: {rule_complexity}")
            print(f"  Updated loss: {final_loss:.6f}")
            print(f"  Basis size: {ogb.t}")
            print()

    # Final predictions (after the loop)
    if len(rules) == 0:
        final_predictions = np.full(n_samples, intercept)
        all_weights = [intercept]
    else:
        all_weights = [intercept] + current_weights
        final_predictions = final_pred  # Use last computed prediction
    
    if verbose:
        print("=" * 60)
        print(f"Final ensemble: {len(rules)} rules")
        print(f"Final loss: {np.mean((y - final_predictions) ** 2):.6f}")
        total_complexity = len(rules) + sum(len(r.support_all(X)) for r in rules)
        print(f"Total complexity: {total_complexity}")
        print()
        
        print("Final Rule Ensemble:")
        print(f"  Intercept: {all_weights[0]:.4f}")
        for i, (rule, weight) in enumerate(zip(rules, all_weights[1:])):
            sign = "+" if weight >= 0 else ""
            print(f"  Rule {i+1}: {sign}{weight:.4f} if {rule.as_conj_str()}")
    return {
        'rules': rules,
        'weights': all_weights,
        'predictions': final_predictions,
        'history': history
    }



def predict_ensemble(X, rules, weights):
    """
    Make predictions using a fitted rule ensemble.
    
    Args:
        X: Feature matrix for prediction
        rules: List of rules from generate_rule_ensemble
        weights: Weights from generate_rule_ensemble (includes intercept)
    
    Returns:
        predictions: Array of predictions
    """
    n_samples = len(X)
    
    if len(rules) == 0:
        return np.full(n_samples, weights[0])
    
    predictions = np.full(n_samples, weights[0])  # Start with intercept
    
    for rule, weight in zip(rules, weights[1:]):
        rule_support_indices = rule.support_all(X)  
        rule_output = np.zeros(n_samples)           
        rule_output[rule_support_indices] = 1.0 
        # rule_output = rule.support_all(X).astype(float)
        predictions += weight * rule_output
    
    return predictions

In [6]:
ogb_objective = OrthogonalGradientBoosting(y_4_100, epsilon=1e-6)
res, val, stats = greedy_maximization(x_4_100, ogb_objective)
print("Efficient mode:", res.as_conj_str(), "Value:", val, "Support size:", len(res.support_all(x_4_100)))

Efficient mode: x4 <= -0.892 & x4 >= -1.574 Value: 3.3110924302465246 Support size: 13


In [7]:
np.random.seed(42)
n_samples = 200
n_features = 4

X = np.random.randn(n_samples, n_features)

# Create target with interpretable structure
y_true = (
    1.5 +                                    # Intercept
    2.0 * (X[:, 0] > 0.5) +                 # Rule 1
    -1.5 * (X[:, 1] < -0.3) +               # Rule 2
    1.0 * ((X[:, 0] > 0) & (X[:, 2] < 0)) + # Rule 3
    0.3 * np.random.randn(n_samples)        # Noise
)

# Split data
n_train = 150
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y_true[:n_train], y_true[n_train:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print()

# Generate rule ensemble
ensemble = generate_rule_ensemble(
    X_train, y_train,
    max_rules=6,
    max_depth=3,
    epsilon=1e-6,
    lambda_reg=0.01,
    verbose=True
)

# Make predictions
train_pred = predict_ensemble(X_train, ensemble['rules'], ensemble['weights'])
test_pred = predict_ensemble(X_test, ensemble['rules'], ensemble['weights'])

train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)

print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Training MSE: {train_mse:.6f}")
print(f"Test MSE: {test_mse:.6f}")
print(f"Number of rules: {len(ensemble['rules'])}")


Training samples: 150
Test samples: 50

Generating rule ensemble: max_rules=6, max_depth=3
Initial loss (intercept only): 2.198177
Initial intercept: 1.7308

Iteration 1:
  Current loss: 2.198177
  Residual norm: 18.158372
  Current Rule Ensemble (Iteration 1):
    Intercept: 1.2082
    Rule 1: +2.7995 if x1 >= 0.508 & x2 >= -0.268

  Rule: x1 >= 0.508 & x2 >= -0.268
  Objective: 12.842050
  Weight: 2.7995
  Support size: 28
  Updated loss: 0.851549
  Basis size: 1

Iteration 2:
  Current loss: 0.851549
  Residual norm: 11.301876
  Current Rule Ensemble (Iteration 2):
    Intercept: 1.1149
    Rule 1: +2.0442 if x1 >= 0.508 & x2 >= -0.268
    Rule 2: +1.4645 if x1 >= 0.378 & x3 <= 0.030

  Rule: x1 >= 0.378 & x3 <= 0.030
  Objective: 5.330230
  Weight: 1.4645
  Support size: 24
  Updated loss: 0.649333
  Basis size: 2

Iteration 3:
  Current loss: 0.649333
  Residual norm: 9.869138
  Current Rule Ensemble (Iteration 3):
    Intercept: 1.2176
    Rule 1: +1.9763 if x1 >= 0.508 & x2 >= -